# Lab 5 - Text Representation

Tasks for this lab: cosine similarity and TF-IDF on small sets of sentences, then a skip-gram Word2Vec model trained on the Simpsons script lines.

Libraries used: pandas, scikit-learn, nltk, gensim.

Install the libraries first if they are missing: `pip install pandas scikit-learn nltk gensim ipywidgets`

In [1]:
import warnings
warnings.filterwarnings("ignore")

import re

import gensim
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

_ = nltk.download("punkt", quiet=True)
_ = nltk.download("punkt_tab", quiet=True)

## Task 1: Cosine Similarity

I turn the four sentences into TF-IDF vectors and then use cosine similarity to compare every sentence with every other one.

In [2]:
sentences = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(sentences)

similarity = cosine_similarity(vectors)
df_similarity = pd.DataFrame(similarity, index=sentences, columns=sentences).round(3)
df_similarity

,This is the first document.,This document is the second document.,And this is the third one.,Is this the first document?
This is the first document.,1.000,0.647,0.308,1.000
This document is the second document.,0.647,1.000,0.225,0.647
And this is the third one.,0.308,0.225,1.000,0.308
Is this the first document?,1.000,0.647,0.308,1.000


In [3]:
# most similar pair (ignoring a sentence compared with itself)
best = (0, 0, -1)
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        if similarity[i][j] > best[2]:
            best = (i, j, similarity[i][j])

print("Most similar pair (%.3f):" % best[2])
print(" -", sentences[best[0]])
print(" -", sentences[best[1]])

Most similar pair (1.000):
 - This is the first document.
 - Is this the first document?


## Task 2: TF-IDF

TF-IDF on the three sentences. A word gets a high score when it appears a lot in one sentence but not in the others, so these are the important words of that sentence.

In [4]:
docs = [
    "data science is one of the most important fields of science",
    "this is one of the best data science courses",
    "data scientists analyze data",
]

tfidf = TfidfVectorizer()
result = tfidf.fit_transform(docs)

df_tfidf = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out()).round(3)
df_tfidf.index = ["sentence 1", "sentence 2", "sentence 3"]
df_tfidf

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
sentence 1,0.000,0.0,0.0,0.190,0.321,0.321,0.244,0.321,0.488,0.244,0.488,0.000,0.244,0.0
sentence 2,0.000,0.4,0.4,0.236,0.000,0.000,0.304,0.000,0.304,0.304,0.304,0.000,0.304,0.4
sentence 3,0.543,0.0,0.0,0.641,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.543,0.000,0.0


In [5]:
# top 3 words of each sentence
words = tfidf.get_feature_names_out()
for i in range(len(docs)):
    row = result[i].toarray()[0]
    top = row.argsort()[::-1][:3]
    print(f"sentence {i + 1}:", [(words[t], round(float(row[t]), 3)) for t in top])

sentence 1: [('of', 0.488), ('science', 0.488), ('important', 0.321)]
sentence 2: [('this', 0.4), ('best', 0.4), ('courses', 0.4)]
sentence 3: [('data', 0.641), ('scientists', 0.543), ('analyze', 0.543)]


## Word2vec: Simpsons dataset

## Task 3: Preprocessing and skip-gram model

I load `simpsons_script_lines.csv` (it is in the `dataset` folder), use the `spoken_words` column, clean it with the `clean_text` function from the lab, tokenize it and train the skip-gram Word2Vec model.

In [6]:
df = pd.read_csv("dataset/simpsons_script_lines.csv")
print(df.shape)
print("missing spoken_words:", df["spoken_words"].isna().sum())

df = df.dropna(subset=["spoken_words"])
df[["raw_character_text", "spoken_words"]].head()

(158314, 2)
missing spoken_words: 26459


,raw_character_text,spoken_words
0,Miss Hoover,"No, actually, it was a little of both. Sometim..."
1,Lisa Simpson,Where's Mr. Bergstrom?
2,Miss Hoover,I don't know. Although I'd sure like to talk t...
3,Lisa Simpson,That life is worth living.
4,Edna Krabappel-Flanders,The polls will be open from now until the end ...


In [7]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", "", text)
    text = re.sub(r"[)(,”“.’$-]", "", text)
    return text

df["clean"] = df["spoken_words"].apply(clean_text)
df[["spoken_words", "clean"]].head()

,spoken_words,clean
0,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...
1,Where's Mr. Bergstrom?,where's mr bergstrom?
2,I don't know. Although I'd sure like to talk t...,i don't know although i'd sure like to talk to...
3,That life is worth living.,that life is worth living
4,The polls will be open from now until the end ...,the polls will be open from now until the end ...


In [8]:
tokens = [word_tokenize(line) for line in df["clean"]]
tokens = [t for t in tokens if len(t) > 0]
print("number of lines:", len(tokens))
print(tokens[0])

number of lines: 131839
['no', 'actually', 'it', 'was', 'a', 'little', 'of', 'both', 'sometimes', 'when', 'a', 'disease', 'is', 'in', 'all', 'the', 'magazines', 'and', 'all', 'the', 'news', 'shows', 'it', "'s", 'only', 'natural', 'that', 'you', 'think', 'you', 'have', 'it']


In [9]:
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)
print("vocabulary size:", len(Skip_gram_model.wv))

vocabulary size: 44175


## Task 4: most_similar()

In [10]:
for name in ["homer", "marge", "bart"]:
    print(f"words similar to '{name}':")
    for word, score in Skip_gram_model.wv.most_similar(name):
        print(f"   {word:15s} {score:.3f}")
    print()

words similar to 'homer':
   abe             0.872
   marge           0.843
   grampa          0.837
   bart            0.828
   eliza           0.824
   ned             0.822
   apu             0.817
   barney          0.815
   mindy           0.811
   agnes           0.811

words similar to 'marge':
   abe             0.846
   sweetie         0.846
   homer           0.843
   sweetheart      0.828
   lisa            0.824
   allison         0.820
   agnes           0.816
   midge           0.813
   honey           0.813
   becky           0.811

words similar to 'bart':
   milhouse        0.873
   lisa            0.868
   grampa          0.847
   sweetie         0.838
   abe             0.835
   eliza           0.831
   homer           0.828
   maggie          0.821
   sweetheart      0.818
   kirk            0.817



## Task 5: doesnt_match()

In [11]:
groups = [
    ["jimbo", "milhouse", "kearney"],
    ["nelson", "bart", "milhouse"],
    ["homer", "patty", "selma"],
]

for g in groups:
    print(g, "-> odd one out:", Skip_gram_model.wv.doesnt_match(g))

['jimbo', 'milhouse', 'kearney'] -> odd one out: milhouse
['nelson', 'bart', 'milhouse'] -> odd one out: nelson
['homer', 'patty', 'selma'] -> odd one out: homer


## Notes on the results

- **Cosine similarity:** the first and fourth sentences have the same words (only the order is different), so their similarity is 1.0. The third sentence is the most different from the others.
- **TF-IDF:** the important words are the ones that are special to one sentence, like `scientists` and `analyze` in sentence 3. Words like `of` and `science` get a high score in sentence 1 because they appear twice there.
- **Word2Vec:** the similar words for `homer`, `marge` and `bart` are mostly other characters from the show (family and town people), and `marge` also gets words like `sweetie` and `honey`. The odd-one-out answers look reasonable: `milhouse` is not a bully like Jimbo and Kearney, and `homer` is not one of the sisters Patty and Selma. For `nelson`, `bart`, `milhouse` the model picked `nelson`.
- The Word2Vec numbers change a little every time you run it because the training starts from random weights.